In [ ]:
# ------------------------------------------------------------------
# BƯỚC 1: CÀI ĐẶT MÔI TRƯỜNG
# ------------------------------------------------------------------
# Cài ultralytics và thư viện hỗ trợ export NCNN
!pip install ultralytics -q
!pip install ncnn -q


import os
from ultralytics import YOLO

# ------------------------------------------------------------------
# BƯỚC 2: TẢI DATASET (Thay API Key của bạn vào)
# ------------------------------------------------------------------
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="tmdNLuOJYbHhzuJbaua8")
project = rf.workspace("vietnamese-german-university-nt2sj").project("pochole")
version = project.version(7)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...


In [ ]:
# ------------------------------------------------------------------
# BƯỚC 3: CONFIG TRAINING "HARDCORE"
# ------------------------------------------------------------------
# Load model Nano pre-trained (Transfer Learning)
model = YOLO('yolov8n-seg.pt')

# 1. imgsz=640: Train ở độ phân giải tiêu chuẩn.
# 2. epochs=150:
# 3. close_mosaic=20: Tắt augmentation "Mosaic" ở 20 epoch cuối

print(">>> BẮT ĐẦU TRAINING VỚI CẤU HÌNH TỐI ƯU...")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    imgsz=640,
    epochs=150,
    batch=16,
    patience=30,
    device=0,
    workers=8,

    # Optimizer settings
    optimizer='auto',   # Ultralytics tự chọn
    lr0=0.01,           # Initial learning rate tiêu chuẩn
    lrf=0.01,           # Final learning rate

    # Advanced Augmentation (Ghi đè config mặc định để tăng độ khó)
    # Vì ổ gà ngoài đường rất đa dạng ánh sáng/màu sắc
    hsv_h=0.015,        # Chỉnh nhẹ Hue
    hsv_s=0.7,          # Tăng variation về độ bão hòa màu
    hsv_v=0.4,          # Tăng variation về độ sáng

    # Tắt Mosaic augmentation ở cuối để fine-tune
    close_mosaic=20,

    project='Pi4_Pothole_Master',
    name='yolov8n_seg_ncnn_ready',
    exist_ok=True
)

>>> BẮT ĐẦU TRAINING VỚI CẤU HÌNH TỐI ƯU...
Ultralytics 8.3.252 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Pochole-7/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_seg_ncnn_ready, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overla

In [ ]:
import shutil
import os
from google.colab import files
import datetime


project_name = 'Pi4_Pothole_Master'
run_name = 'yolov8n_seg_ncnn_ready'

# Đường dẫn thư mục kết quả YOLO
source_dir = f"{project_name}/{run_name}"
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
zip_filename = f"KetQua_YOLO_{timestamp}.zip"

# ==============================================================================
# XỬ LÝ NÉN VÀ TẢI VỀ
# ==============================================================================
if os.path.exists(source_dir):
    print(f">>> Tìm thấy thư mục kết quả: {source_dir}")
    print(f">>> Đang nén dữ liệu... (Vui lòng đợi vài giây)")

    # 1. Nén thư mục thành file .zip
    shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', source_dir)

    print(f">>> Đã nén xong thành file: {zip_filename}")
    print(f">>> Đang gửi lệnh tải về trình duyệt...")

    # 2. Lệnh kích hoạt tải về máy tính (Local)
    files.download(zip_filename)

    print("-" * 50)
    print("✅ ĐÃ GỬI LỆNH TẢI!")
    print("Lưu ý: Giữ tab trình duyệt mở và đảm bảo máy tính không bị Sleep.")
    print("-" * 50)

else:
    print(f"❌ LỖI: Không tìm thấy thư mục '{source_dir}'.")
    print("Bạn hãy kiểm tra lại xem tên 'project' và 'name' trong lệnh train có đúng không.")

>>> Tìm thấy thư mục kết quả: Pi4_Pothole_Master/yolov8n_seg_ncnn_ready
>>> Đang nén dữ liệu... (Vui lòng đợi vài giây)
>>> Đã nén xong thành file: KetQua_YOLO_20260110_2006.zip
>>> Đang gửi lệnh tải về trình duyệt...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

--------------------------------------------------
✅ ĐÃ GỬI LỆNH TẢI!
Lưu ý: Giữ tab trình duyệt mở và đảm bảo máy tính không bị Sleep.
--------------------------------------------------


In [ ]:

from ultralytics import YOLO

# 1. Tìm đường dẫn file best.pt (Nhìn trong log của bạn nó nằm ở đây)
# Lưu ý: Nếu bạn chạy nhiều lần, folder có thể là Pi4_Pothole_Master2, 3...
# Hãy kiểm tra cột bên trái Colab để chắc chắn tên folder.
best_weight_path = '/content/Pi4_Pothole_Master/yolov8n_seg_ncnn_ready/weights/best.pt'

print(f"Loading best model from: {best_weight_path}")
best_model = YOLO(best_weight_path)

# 2. Validate lại để bạn yên tâm (Sẽ ra số ~0.73 như log training)
print("\n>>> ĐANG KIỂM TRA LẠI MODEL TỐT NHẤT...")
metrics = best_model.val(data=f"{dataset.location}/data.yaml")
print(f"--- KẾT QUẢ THỰC TẾ ---")
print(f"mAP50 (Độ chính xác chuẩn): {metrics.seg.map50:.4f}")
# Bạn sẽ thấy con số này nhảy lên ~0.7 ngay!

# 3. EXPORT SANG NCNN (CHO RASPBERRY PI 4)
print("\n>>> ĐANG EXPORT SANG NCNN...")
# imgsz=640 là chuẩn
export_path = best_model.export(format='ncnn', imgsz=640)
print(f"Done! File model nằm tại: {export_path}")

# 4. Nén và tải về
!zip -r ncnn_model_final.zip {export_path}

Loading best model from: /content/Pi4_Pothole_Master/yolov8n_seg_ncnn_ready/weights/best.pt

>>> ĐANG KIỂM TRA LẠI MODEL TỐT NHẤT...
Ultralytics 8.3.252 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1028.3±429.8 MB/s, size: 47.3 KB)
val: Scanning /content/Pochole-7/valid/labels.cache... 189 images, 55 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 189/189 56.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.2it/s 9.6s
                   all        189        245      0.765       0.62      0.717      0.429      0.768      0.637      0.726       0.41
Speed: 6.8ms preprocess, 12.9ms inference, 0.0ms loss, 8.6ms postprocess per image
Results saved to /content/runs/segment/val
--- KẾT QUẢ THỰC TẾ ---
mAP50 (Độ chính xác chuẩ

In [ ]:
++
